In [5]:
from __future__ import annotations

# --- Notebook cell 0 (date + scrape helpers) ---
import requests
import trafilatura
import json
import re
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from urllib.parse import urlparse

# Try to import Newspaper3k as an optional fallback extractor
try:
    from newspaper import Article as _NPArticle
    _HAS_NEWSPAPER = True
except Exception:
    _HAS_NEWSPAPER = False


# ------------------ DATE EXTRACTION ------------------ #

_MONTHS = (
    "January|February|March|April|May|June|July|August|September|October|November|December"
)


def fallback_date_from_url(url: str) -> str:
    """
    Best-effort fallback date extraction from URL patterns.
    Returns ISO date 'YYYY-MM-DD' or 'unknown'.

    Supports:
      - /YYYY/MM/DD/  (common on WordPress)
      - /afp/YYMMDD... (AFP wire URLs)
    """
    try:
        path = urlparse(url).path or ""
    except Exception:
        return "unknown"

    m = re.search(r"/(20\d{2})/(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])/", path)
    if m:
        yyyy, mm, dd = m.group(1), m.group(2), m.group(3)
        return f"{yyyy}-{mm}-{dd}"

    m = re.search(r"/afp/(\d{2})(\d{2})(\d{2})", path)
    if m:
        yy, mm, dd = m.group(1), m.group(2), m.group(3)
        return f"20{yy}-{mm}-{dd}"

    return "unknown"


def parse_date_str(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return "unknown"
    try:
        dt = dtparser.parse(s, fuzzy=True)
        return dt.date().isoformat()
    except Exception:
        return "unknown"


def extract_date_from_html(html_text: str) -> str:
    """Try common meta tags and time elements."""

    try:
        soup = BeautifulSoup(html_text or "", "html.parser")
    except Exception:
        return "unknown"

    meta_names = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "pubdate"}),
        ("meta", {"name": "publishdate"}),
        ("meta", {"name": "publish_date"}),
        ("meta", {"name": "date"}),
        ("meta", {"name": "Date"}),
        ("meta", {"name": "dc.date"}),
        ("meta", {"name": "DC.date"}),
        ("meta", {"property": "og:updated_time"}),
        ("meta", {"name": "parsely-pub-date"}),
    ]

    for tag, attrs in meta_names:
        el = soup.find(tag, attrs=attrs)
        if el and el.get("content"):
            d = parse_date_str(el.get("content"))
            if d != "unknown":
                return d

    # <time datetime="...">
    t = soup.find("time")
    if t:
        if t.get("datetime"):
            d = parse_date_str(t.get("datetime"))
            if d != "unknown":
                return d
        # sometimes time tag contains the date
        d = parse_date_str(t.get_text(" ", strip=True))
        if d != "unknown":
            return d

    # Regex fallback (e.g. "December 8, 2025")
    m = re.search(rf"({ _MONTHS })\s+\d{{1,2}},\s+20\d{{2}}", soup.get_text(" ", strip=True))
    if m:
        return parse_date_str(m.group(0))

    return "unknown"


def trafilatura_extract(url: str, timeout: int = 20) -> dict:
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"ok": False, "text": "", "title": "", "html": "", "final_url": url, "error": "fetch_url_returned_empty"}

        text = trafilatura.extract(downloaded) or ""
        title = ""
        try:
            meta = trafilatura.metadata.extract_metadata(downloaded)
            title = (meta.title or "") if meta else ""
        except Exception:
            title = ""

        return {"ok": bool(text.strip()), "text": text, "title": title, "html": downloaded, "final_url": url, "error": ""}
    except Exception as e:
        return {"ok": False, "text": "", "title": "", "html": "", "final_url": url, "error": f"{type(e).__name__}: {e}"}

def newspaper_extract(url: str, timeout: int = 20) -> dict:
    """Fallback: Newspaper3k extraction."""

    if not _HAS_NEWSPAPER:
        return {"ok": False, "text": "", "title": "", "html": "", "final_url": url}

    try:
        art = _NPArticle(url)
        art.download()
        art.parse()
        text = (art.text or "")
        title = (art.title or "")
        return {"ok": bool(text.strip()), "text": text, "title": title, "html": "", "final_url": url}
    except Exception:
        return {"ok": False, "text": "", "title": "", "html": "", "final_url": url}